# 03 · Baseline DINOv2-small

Entrena un clasificador por estudio desde el cache, usando pseudo-etiquetas si existen. Un fold suele tardar varias horas; usa una GPU de 48 GB o más. Guarda checkpoint, OOF, métricas y curvas en tu carpeta de runs.

In [ ]:
# CONFIG
RUN_NAME = "dinov2s_baseline_v1"
CACHE_ROOT = "/workspace/data/cache/v1_256"
RAW_ROOT = "/workspace/data/raw"
LABELS_CSV = "/workspace/data/labels/report_labels_v1.csv"
BACKBONE_PATH = "/workspace/models/dinov2-small"
FOLD = 0
FOLDS_TO_RUN = []
IMAGE_SIZE = 224
MAX_SERIES = 5
N_SLICES = 16
BATCH_SIZE = 2
EPOCHS = 5
LR = 1e-4
GRAD_ACCUM = 4
N_WORKERS = 4
FREEZE_EPOCHS = 1
NOTES = "baseline"


In [ ]:
from pathlib import Path
import os
import sys
import warnings
import matplotlib.pyplot as plt
import pandas as pd

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo))
from src.knee.config import LABELS, user_dir
from src.knee.train import train_fold

def read_train_csv(path):
    for encoding in ("utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise RuntimeError(f"No se pudo leer {path}")

gold = read_train_csv(Path(RAW_ROOT) / "train.csv")
if Path(LABELS_CSV).exists():
    labels = pd.read_csv(LABELS_CSV)[["StudyInstanceUID", *LABELS]]
    gold_rows = gold.dropna(subset=LABELS, how="all").set_index("StudyInstanceUID")
    labels = labels.set_index("StudyInstanceUID")
    labels.loc[labels.index.intersection(gold_rows.index), LABELS] = gold_rows.loc[labels.index.intersection(gold_rows.index), LABELS]
    labels = labels.reset_index()
    print(f"Entrenando con {len(labels)} estudios; gold reemplaza pseudo cuando existe.")
    prepared = user_dir() / "runs" / RUN_NAME / "training_labels.csv"
    prepared.parent.mkdir(parents=True, exist_ok=True)
    labels.to_csv(prepared, index=False)
    LABELS_CSV = str(prepared)
else:
    warnings.warn("No existe el archivo de pseudo-etiquetas. Se usarán solo los 58 estudios gold.")
    labels = gold.dropna(subset=LABELS, how="all")[["StudyInstanceUID", *LABELS]]
    fallback = user_dir() / "runs" / RUN_NAME / "gold_only_labels.csv"
    fallback.parent.mkdir(parents=True, exist_ok=True)
    labels.to_csv(fallback, index=False)
    LABELS_CSV = str(fallback)


In [ ]:
folds = FOLDS_TO_RUN or [FOLD]
all_metrics = {}
for fold in folds:
    fold_dir = user_dir() / "runs" / RUN_NAME / f"fold_{fold}"
    cfg = {
        "run_name": RUN_NAME, "run_dir": str(fold_dir), "fold": fold,
        "labels_csv": LABELS_CSV, "cache_index": str(Path(CACHE_ROOT) / "index.csv"),
        "backbone_path": BACKBONE_PATH, "image_size": IMAGE_SIZE,
        "max_series": MAX_SERIES, "n_slices": N_SLICES, "batch_size": BATCH_SIZE,
        "epochs": EPOCHS, "lr": LR, "grad_accum": GRAD_ACCUM,
        "workers": N_WORKERS, "freeze_epochs": FREEZE_EPOCHS,
        "gradient_checkpointing": True, "notes": NOTES,
    }
    all_metrics[fold] = train_fold(cfg)


In [ ]:
for fold, result in all_metrics.items():
    history = pd.DataFrame(result["history"])
    display(pd.DataFrame(result["metrics"]["per_label"].items(), columns=["label", "auc"]))
    history.plot(x="epoch", y=["train_loss", "val_auc_mean"], marker="o", title=f"Fold {fold}")
    plt.show()
